In [2]:
import requests
import pandas as pd
import time

pd.set_option('display.max_columns', None)

API_KEY  = 'Hn90mDfDS74XLWD9AxM_Ty9TEAg9LXqUsApkE76Lim0'
BASE_URL = 'https://api.shovels.ai/v2/permits/search'
HEADERS  = {'X-API-Key': API_KEY}

STATE      = 'FL'
DATE_FROM  = '2026-01-01'
DATE_TO    = '2026-04-30'
SAMPLE_N   = 100

# Florida Shovels Exploration

Exploratory pull of Florida residential permits from the Shovels API (2026 YTD sample) to understand what inspection and person/identity fields are available in the data.

In [3]:
# Pull a sample of residential permits from FL in 2026
r = requests.get(BASE_URL, headers=HEADERS, params={
    'geo_id':        STATE,
    'permit_from':   DATE_FROM,
    'permit_to':     DATE_TO,
    'property_type': 'residential',
    'size':          SAMPLE_N,
    'include_count': 'true',
})
r.raise_for_status()
data = r.json()

total = data.get('total_count', {})
count = total.get('value', '?') if isinstance(total, dict) else total
print(f'Total FL residential permits in period (API estimate): {count}')
print(f'Fetched: {len(data["items"])} records')

df = pd.json_normalize(data['items'])
print(f'\nColumns ({len(df.columns)}):')
for col in df.columns:
    print(f'  {col}')

Total FL residential permits in period (API estimate): 10000
Fetched: 100 records

Columns (45):
  property_census_tract
  property_congressional_district
  property_type
  property_type_detail
  property_legal_owner
  property_owner_type
  property_lot_size
  property_building_area
  property_story_count
  property_unit_count
  property_year_built
  property_assess_market_value
  id
  number
  description
  jurisdiction
  job_value
  type
  subtype
  fees
  status
  file_date
  issue_date
  final_date
  start_date
  end_date
  total_duration
  construction_duration
  approval_duration
  inspection_pass_rate
  contractor_id
  tags
  address.street_no
  address.street
  address.city
  address.county
  address.zip_code
  address.zip_code_ext
  address.state
  address.jurisdiction
  address.latlng
  geo_ids.address_id
  geo_ids.city_id
  geo_ids.county_id
  geo_ids.jurisdiction_id


In [4]:
# Look for inspection-related and person/ID fields
inspection_cols = [c for c in df.columns if any(x in c.lower() for x in
    ['inspect', 'duration', 'approv', 'issued', 'final', 'complet', 'status', 'date'])]
person_cols = [c for c in df.columns if any(x in c.lower() for x in
    ['name', 'owner', 'contact', 'applicant', 'contractor', 'agent', 'inspector', 'id'])]

print('=== Inspection / timeline fields ===')
for c in inspection_cols:
    print(f'  {c}')
    print(f'    {df[c].value_counts(dropna=False).head(3).to_dict()}')

print()
print('=== Person / ID fields ===')
for c in person_cols:
    print(f'  {c}')
    print(f'    {df[c].value_counts(dropna=False).head(3).to_dict()}')

=== Inspection / timeline fields ===
  status
    {'active': 42, 'in_review': 42, None: 8}
  file_date
    {'2026-04-30': 93, None: 7}
  issue_date
    {'2026-04-30': 86, None: 10, '2026-05-05': 2}
  final_date
    {None: 88, '2026-11-04': 2, '2026-05-11': 2}
  start_date
    {'2026-04-30': 100}
  end_date
    {'2026-04-30': 85, '2026-11-04': 2, '2026-05-05': 2}
  total_duration
    {0: 85, 188: 2, 5: 2}
  construction_duration
    {nan: 89, 184.0: 2, 11.0: 2}
  approval_duration
    {0.0: 79, nan: 17, 5.0: 2}
  inspection_pass_rate
    {None: 100}

=== Person / ID fields ===
  property_legal_owner
    {None: 25, 'Americas 2000 Llc': 3, 'Carle Joyce S Living Trust; Carle Joyce S Tre': 2}
  property_owner_type
    {'individual': 53, None: 25, 'company_owned': 22}
  id
    {'000249f80fc35baa': 1, '05d2a27ffd6397ba': 1, '06d9fb5cb9123933': 1}
  contractor_id
    {None: 16, '8kz8JkUQ3D': 2, 'EqronM9MNG': 2}
  geo_ids.address_id
    {None: 9, 'CXY-4dqcF-Y': 2, 'CQsGvxgInDE': 2}
  geo_ids.ci

In [5]:
# Print one raw record in full
print('=== Full first record ===')
for k, v in data['items'][0].items():
    print(f'  {k:<45} {v}')

=== Full first record ===
  property_census_tract                         120710019.232000
  property_congressional_district               19
  property_type                                 residential
  property_type_detail                          Condominium Unit
  property_legal_owner                          Chris Jannelli
  property_owner_type                           individual
  property_lot_size                             6692
  property_building_area                        774
  property_story_count                          2
  property_unit_count                           1
  property_year_built                           1982
  property_assess_market_value                  641100
  id                                            000249f80fc35baa
  number                                        COM2022-02793-R04
  description                                   Add turnkey scope of work to permit for units 112, 113, 115 and 116
  jurisdiction                                  LEE